# Figure 10: SAM 2 Figures for All ImageNet Samples

This notebook loops over every image in `benchmarks/vggnet16_benchmark2022/imagenet-sample`,
segments each image with SAM 2 using grid prompts, and saves one figure per image.

Each saved filename is:

- the original image stem
- plus the selected SAM 2 score

Example output name:
`n02108915_French_bulldog__score_0p9455.png`

If you only want a small test run first, set `MAX_IMAGES` to a small integer.

In [26]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.patches import Rectangle
from PIL import Image, ImageOps


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "scripts" / "sam2_utils.py").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate the XAIV project root from the current working directory."
    )


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.sam2_utils import load_sam2_predictor, run_segmentation_model


IMAGE_DIR = PROJECT_ROOT / "benchmarks" / "vggnet16_benchmark2022" / "imagenet-sample"
OUTPUT_DIR = PROJECT_ROOT / "plots" / "Figure_10" / "saved_figures"

MAX_IMAGES = None  # Set an integer like 10 for a quick test run.
MAX_LONG_SIDE = 768
GRID_SIZE = 6
SCORE_THRESHOLD = 0.0
MAX_SEGMENTS = 6
MIN_AREA_RATIO = 0.01
MAX_AREA_RATIO = 0.85
MODEL_NAME = "facebook/sam2-hiera-large"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SAVE_DPI = 220

assert IMAGE_DIR.exists(), f"Image directory not found: {IMAGE_DIR}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

image_files = sorted(
    p
    for p in IMAGE_DIR.iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
)
assert image_files, f"No images found in {IMAGE_DIR}"

if MAX_IMAGES is not None:
    image_files = image_files[:MAX_IMAGES]

plt.rcParams.update(
    {
        "figure.facecolor": "#f4f1ea",
        "axes.facecolor": "#fffdf8",
        "axes.edgecolor": "#d8d2c4",
        "savefig.facecolor": "#f4f1ea",
        "font.size": 11,
        "axes.titlesize": 13,
    }
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Images to process: {len(image_files)}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"SAM 2 device: {DEVICE}")

Project root: /Users/zd3504phd/Desktop/XAIV
Images to process: 1000
Output directory: /Users/zd3504phd/Desktop/XAIV/plots/Figure_10/saved_figures
SAM 2 device: cpu


In [27]:
def pretty_label(path: Path) -> str:
    stem = path.stem
    if "_" not in stem:
        return stem
    return stem.split("_", 1)[1].replace("_", " ")


def resize_for_display(image: Image.Image, max_long_side: int = MAX_LONG_SIDE) -> Image.Image:
    image = ImageOps.exif_transpose(image).convert("RGB")
    width, height = image.size
    scale = min(1.0, max_long_side / max(width, height))
    if scale == 1.0:
        return image
    new_size = (max(1, round(width * scale)), max(1, round(height * scale)))
    return image.resize(new_size, Image.Resampling.LANCZOS)


def choose_display_segment(
    segments,
    min_area_ratio: float = MIN_AREA_RATIO,
    max_area_ratio: float = MAX_AREA_RATIO,
):
    ranked_segments = []
    for idx, segment in enumerate(segments, start=1):
        mask = np.asarray(segment["mask"], dtype=bool)
        ranked_segments.append(
            {
                **segment,
                "mask": mask,
                "rank": idx,
                "area_ratio": float(mask.mean()),
            }
        )

    for segment in ranked_segments:
        if min_area_ratio <= segment["area_ratio"] <= max_area_ratio:
            return segment, ranked_segments
    return ranked_segments[0], ranked_segments


def make_overlay(
    image_np: np.ndarray,
    mask: np.ndarray,
    color=(0, 214, 143),
    alpha: float = 0.42,
) -> np.ndarray:
    overlay = image_np.astype(np.float32).copy()
    color_np = np.array(color, dtype=np.float32)
    overlay[mask] = (1.0 - alpha) * overlay[mask] + alpha * color_np
    return np.clip(overlay, 0, 255).astype(np.uint8)


def top_score_lines(ranked_segments, top_k: int = 5):
    lines = []
    for segment in ranked_segments[:top_k]:
        lines.append(
            f"#{segment['rank']}: score={segment['score']:.3f}, area={segment['area_ratio'] * 100:.1f}%"
        )
    return lines


def format_score_for_filename(score: float, decimals: int = 4) -> str:
    return f"{score:.{decimals}f}".replace(".", "p")


def build_output_path(image_path: Path, score: float) -> Path:
    filename = f"{image_path.stem}__score_{format_score_for_filename(score)}.png"
    return OUTPUT_DIR / filename


def render_segment_figure(
    image_path: Path,
    image_np: np.ndarray,
    display_segment,
    ranked_segments,
):
    display_mask = display_segment["mask"]
    overlay_np = make_overlay(image_np, display_mask)

    fig, axes = plt.subplots(
        1,
        4,
        figsize=(18, 5.4),
        gridspec_kw={"width_ratios": [1.0, 1.0, 1.0, 0.95]},
        constrained_layout=True,
    )

    ax_original, ax_overlay, ax_mask, ax_info = axes
    for ax in (ax_original, ax_overlay, ax_mask):
        ax.set_xticks([])
        ax.set_yticks([])

    ax_original.imshow(image_np)
    ax_original.set_title("Original image")

    ax_overlay.imshow(overlay_np)
    x0, y0, x1, y1 = display_segment["bbox"]
    ax_overlay.add_patch(
        Rectangle(
            (x0, y0),
            max(1, x1 - x0 + 1),
            max(1, y1 - y0 + 1),
            fill=False,
            linewidth=2.2,
            edgecolor="#00a86b",
        )
    )
    ax_overlay.contour(
        display_mask.astype(float),
        levels=[0.5],
        colors=["#083d2b"],
        linewidths=1.5,
    )
    ax_overlay.set_title(f"SAM 2 overlay | score={display_segment['score']:.3f}")

    ax_mask.imshow(display_mask, cmap="gray")
    ax_mask.set_title("Binary mask")

    ax_info.set_axis_off()
    ax_info.set_facecolor("#fffaf0")
    info_lines = [
        "SAM 2 summary",
        "",
        f"File: {image_path.name}",
        f"Class: {pretty_label(image_path)}",
        f"Device: {DEVICE}",
        f"Grid size: {GRID_SIZE} x {GRID_SIZE}",
        f"Mask rank: #{display_segment['rank']}",
        f"Mask score: {display_segment['score']:.4f}",
        f"Mask area: {display_segment['area_ratio'] * 100:.2f}%",
        f"BBox: {display_segment['bbox']}",
        "",
        "Top masks:",
        *top_score_lines(ranked_segments),
    ]
    ax_info.text(
        0.03,
        0.98,
        "\n".join(info_lines),
        va="top",
        ha="left",
        family="monospace",
        fontsize=10.5,
        color="#2d261c",
    )

    fig.suptitle(
        f"{pretty_label(image_path)} image, segmented with SAM 2",
        fontsize=16,
        y=1.03,
    )
    return fig

In [28]:
SEG_CFG = {
    "model_name": MODEL_NAME,
    "device": DEVICE,
}

predictor = load_sam2_predictor(SEG_CFG)
print("Loaded SAM 2 predictor.")

[SAM2] Using device: cpu
Loaded SAM 2 predictor.


In [29]:
saved_paths = []
results = []
failed_images = []

for idx, image_path in enumerate(image_files, start=1):
    try:
        with Image.open(image_path) as pil_image:
            image = resize_for_display(pil_image)

        image_np = np.asarray(image)
        image_np_float = image_np.astype(np.float32) / 255.0

        segments = run_segmentation_model(
            predictor,
            image_np_float,
            grid_size=GRID_SIZE,
            score_threshold=SCORE_THRESHOLD,
            max_segments=MAX_SEGMENTS,
        )

        if not segments:
            print(
                f"[{idx}/{len(image_files)}] skipped {image_path.name} because SAM 2 returned no segments"
            )
            continue

        display_segment, ranked_segments = choose_display_segment(segments)
        output_path = build_output_path(image_path, display_segment["score"])
        fig = render_segment_figure(image_path, image_np, display_segment, ranked_segments)
        fig.savefig(output_path, dpi=SAVE_DPI, bbox_inches="tight")
        plt.close(fig)

        saved_paths.append(output_path)
        results.append(
            {
                "image_name": image_path.name,
                "saved_name": output_path.name,
                "score": float(display_segment["score"]),
                "area_ratio": float(display_segment["area_ratio"]),
                "bbox": display_segment["bbox"],
            }
        )
        print(f"[{idx}/{len(image_files)}] saved {output_path.name}")
    except Exception as exc:
        failed_images.append((image_path.name, str(exc)))
        print(f"[{idx}/{len(image_files)}] failed {image_path.name}: {exc}")

print(f"Saved {len(saved_paths)} figures to {OUTPUT_DIR}")
print(f"Failed images: {len(failed_images)}")

[SAM2] Found 3 segments after filtering
[1/1000] saved n01440764_tench__score_0p4852.png
[SAM2] Found 3 segments after filtering
[2/1000] saved n01443537_goldfish__score_0p6292.png
[SAM2] Found 3 segments after filtering
[3/1000] saved n01484850_great_white_shark__score_0p2698.png
[SAM2] Found 3 segments after filtering
[4/1000] saved n01491361_tiger_shark__score_0p5060.png
[SAM2] Found 3 segments after filtering
[5/1000] saved n01494475_hammerhead__score_0p8975.png
[SAM2] Found 3 segments after filtering
[6/1000] saved n01496331_electric_ray__score_0p4592.png
[SAM2] Found 3 segments after filtering
[7/1000] saved n01498041_stingray__score_0p9658.png
[SAM2] Found 3 segments after filtering
[8/1000] saved n01514668_cock__score_0p8784.png
[SAM2] Found 3 segments after filtering
[9/1000] saved n01514859_hen__score_0p6795.png
[SAM2] Found 3 segments after filtering
[10/1000] saved n01518878_ostrich__score_0p2950.png
[SAM2] Found 3 segments after filtering
[11/1000] saved n01530575_bramblin

KeyboardInterrupt: 

In [ ]:
if results:
    print("First saved files:")
    for row in results[:10]:
        print(
            f"- {row['saved_name']} | score={row['score']:.4f} | area={row['area_ratio'] * 100:.2f}%"
        )

    preview_path = saved_paths[0]
    with Image.open(preview_path) as preview_image:
        preview_np = np.asarray(preview_image)
    plt.figure(figsize=(16, 5))
    plt.imshow(preview_np)
    plt.axis("off")
    plt.title(f"Preview: {preview_path.name}")
    plt.show()
else:
    print("No figures were saved.")

if failed_images:
    print("First failed files:")
    for image_name, error_text in failed_images[:10]:
        print(f"- {image_name}: {error_text}")